# Email Negotiation — GRPO Training with OpenEnv

**Branch**: `Email_negotiation`  
**Repo**: `https://github.com/akshaypulla/deal_room`

**What this environment is**:
A Two-LLM email negotiation system where a seller LLM (Qwen2.5-3B-Instruct) learns
to close enterprise deals by sending emails to a buying committee (Legal, Finance, CTO, etc.).

**Key design**:
- Buyer stakeholders are frozen LLM agents with persistent memory
- `progress_score = S^0.4 × D^0.2 × G^0.2 × R^0.2` gates all rewards
- 3-layer reward extraction (keyword → LLM → code) — LLM never directly produces reward
- Anti-gaming: policy constraints + diminishing returns + CTA validation

**Grading alignment**:
- Training evidence: reward curves improving over steps (shown in cell 13)
- 20% training: `openenv_reward()` connects TRL GRPOTrainer to the Space
- 30% storytelling: multiplicative progress gating, multi-stakeholder memory, CC causal signal

---

## CELL 1 — Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q 'trl>=0.12.0' bitsandbytes peft accelerate datasets scipy numpy matplotlib
!pip install -q openenv-core[core] openai 2>&1 | tail -3



## CELL 2 — Clone the Email_negotiation branch

The environment lives in `email_negotiation/` inside the `Email_negotiation` branch.

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_URL  = 'https://github.com/akshaypulla/deal_room.git'
BRANCH    = 'Email_negotiation'
TARGET    = Path('/content/deal_room')

if TARGET.exists():
    print(f'DealRoom already at {TARGET}')
    subprocess.run(['git', '-C', str(TARGET), 'pull', '--ff-only'],
                   capture_output=True, timeout=30)
    print('  Updated to latest.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(TARGET)],
                   capture_output=True, check=True)
    print(f'Cloned {BRANCH} branch to {TARGET}')

# Point sys.path at the email_negotiation subdirectory so 'from models import ...'
# resolves to email_negotiation/models.py (not the root deal_room/models.py)
EMAIL_NEG_DIR = TARGET / 'email_negotiation'
sys.path.insert(0, str(EMAIL_NEG_DIR))
os.chdir(EMAIL_NEG_DIR)
print(f'sys.path[0]: {sys.path[0]}')
print(f'cwd: {os.getcwd()}')



## CELL 3 — Verify environment and reward architecture

Check that the environment loads correctly and the reward weights match the plan.

In [ ]:
from server.environment import EmailNegotiationEnvironment
from server.email_env.progress_score import STAKEHOLDER_WEIGHTS
from server.email_env.email_negotiation import EmailNegotiationCore
from server.email_env.reward_extractor import POSITIVE_MARKERS, NEGATIVE_MARKERS
from server.email_env.anti_gaming import AntiGamingValidator

print('=== Stakeholder Weights (S component of progress_score) ===')
for sid, w in sorted(STAKEHOLDER_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f'  {sid:<15}: {w:.2f}')

print('\n=== Email Action Space ===')
print('intents:    address_concern | offer_document | make_concession | escalate_to_exec | group_proposal | walkaway')
print('targets:    Legal | Finance | CTO | TechLead | Procurement | Operations | ExecSponsor')
print('tones:      formal | reassuring | urgent')
print('documents: DPA | roi_model | security_cert | implementation_timeline | vendor_packet')
print('CC:         max 2 (causal influence signal)')

env = EmailNegotiationEnvironment()
obs = env.reset()
print(f'\n=== Environment Reset ===')
print(f'Deal stage:    {obs.deal_stage}')
print(f'Progress:      {obs.progress_score}')
print(f'Round:         {obs.round_number}/{obs.max_rounds}')
print(f'Unresolved:    {obs.unresolved_concerns}')
print(f'Inbox summary (first 400 chars):')
print(obs.inbox_summary[:400])



## CELL 4 — CRITICAL: Verify reward variance before training

**This is the single most important check before any GRPO training run.**

If reward variance is near-zero, GRPO advantage = `(r_i - mean) / std` → 0 everywhere → no gradients.

The previous notebook (v3.6) had `reward_std = 0.000133` — training flatlined.
This environment must show `reward_std > 0.02` for GRPO to learn.

In [ ]:
import numpy as np

# Test diverse actions across all intent types
TEST_CASES = [
    {'intent': 'offer_document',  'target': 'Legal',    'include_document': 'DPA',              'tone': 'formal'},
    {'intent': 'offer_document',  'target': 'Finance',  'include_document': 'roi_model',          'tone': 'formal'},
    {'intent': 'address_concern','target': 'CTO',       'include_document': None,                'tone': 'formal'},
    {'intent': 'make_concession','target': 'Legal',    'include_document': None,
     'concession_term': 'price', 'concession_value': 85000, 'tone': 'formal'},
    {'intent': 'escalate_to_exec','target': 'ExecSponsor','include_document': None,             'tone': 'urgent'},
    {'intent': 'offer_document', 'target': 'Legal',    'include_document': 'security_cert',       'tone': 'reassuring'},
    {'intent': 'group_proposal', 'target': 'Legal',    'cc': ['Finance', 'CTO'],                'tone': 'formal'},
    {'intent': 'address_concern','target': 'Finance',   'include_document': None,                'tone': 'reassuring'},
]

rewards = []
print('Reward variance across action types:')
print('-' * 65)
for i, case in enumerate(TEST_CASES):
    e = EmailNegotiationCore(scenario_type='hostile_acquisition', use_buyer_llm=False)
    e.reset()
    result = e.step(case)
    rewards.append(result['reward'])
    print(f'  [{i+1}] {case["intent"]:<20} -> {case["target"]:<12} '
          f'| reward={result["reward"]:+.4f} '
          f'| progress={result["progress_score"]:.3f}')
    e.close()

print('-' * 65)
print(f'mean={np.mean(rewards):.4f}  std={np.std(rewards):.4f}  '
      f'min={np.min(rewards):.4f}  max={np.max(rewards):.4f}')

assert np.std(rewards) > 0.02, \
    f'CRITICAL: reward_std={np.std(rewards):.4f} < 0.02 — GRPO will not learn!'
print('PASS: reward variance is healthy (std > 0.02) — safe to train')



## CELL 5 — System prompt + action parsing

The LLM generates free text. We parse it into a structured `EmailAction`.
The `###` token marks end-of-action.

In [ ]:
SYSTEM_PROMPT = '''You are an AI vendor salesperson negotiating an enterprise software deal.
Your goal: close the deal by addressing buyer concerns, delivering the right documents,
and making strategic concessions — all via email.

OUTPUT EXACTLY ONE action ending with ###.

Action format (one of):
  address_concern <target> <your message> ###
  offer_document  <target> <doc_type> <your message> ###
  make_concession <target> <term>=<value> <your message> ###
  escalate_to_exec <target> <your message> ###
  group_proposal <your message> ###
  walkaway ###

Targets: Legal, Finance, CTO, TechLead, Procurement, Operations, ExecSponsor
Doc types: DPA, roi_model, security_cert, implementation_timeline, vendor_packet
CC: optionally CC 1-2 other stakeholders as a causal influence signal

Examples:
  offer_document Legal DPA Our DPA covers all compliance requirements. ###
  address_concern Finance We have attached our ROI model showing 3-year payback. ###
  make_concession Legal price=85000 We can reduce the price to address budget concerns. ###
  offer_document Finance roi_model Attached is the detailed ROI analysis. ###
  escalate_to_exec ExecSponsor We would like to schedule a call to discuss final terms. ###

Output ONLY the action line ending with ###. Nothing else.'''

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

In [ ]:
import re

# Target name normalization: "cto" -> "CTO", "execsponsor" -> "ExecSponsor", etc.
_TARGET_MAP = {
    'cto': 'CTO',
    'techlead': 'TechLead',
    'execsponsor': 'ExecSponsor',
    'legal': 'Legal',
    'finance': 'Finance',
    'procurement': 'Procurement',
    'operations': 'Operations',
}

def _norm_target(raw: str) -> str:
    """Normalize LLM output target name to canonical form."""
    key = raw.lower().strip()
    return _TARGET_MAP.get(key, raw.strip().title())


def parse_email_action(text: str) -> dict:
    """
    Parse LLM completion text into an EmailAction dict.
    Handles the structured text format the LLM is prompted to output.
    """
    text = text.strip()
    if '###' in text:
        text = text[:text.index('###') + 3]

    # offer_document target doc_type message ###
    m = re.match(r'offer_document\s+(\w+)\s+(DPA|roi_model|security_cert|implementation_timeline|vendor_packet)\s+(.+)', text, re.IGNORECASE)
    if m:
        return {
            'intent': 'offer_document', 'target': _norm_target(m.group(1)),
            'tone': 'formal', 'cc': [],
            'include_document': m.group(2).upper().replace('_', '_'),
            'concession_term': None, 'concession_value': None, 'urgency': 'normal'
        }

    # make_concession target term=value message ###
    m = re.match(r'make_concession\s+(\w+)\s+(\w+)=([\d.]+)\s+(.+)', text, re.IGNORECASE)
    if m:
        return {
            'intent': 'make_concession', 'target': _norm_target(m.group(1)),
            'tone': 'formal', 'cc': [],
            'include_document': None,
            'concession_term': m.group(2).lower(),
            'concession_value': float(m.group(3)),
            'urgency': 'normal'
        }

    # escalate_to_exec target message ###
    m = re.match(r'escalate_to_exec\s+(\w+)\s+(.+)', text, re.IGNORECASE)
    if m:
        return {
            'intent': 'escalate_to_exec', 'target': _norm_target(m.group(1)),
            'tone': 'urgent', 'cc': [],
            'include_document': None,
            'concession_term': None, 'concession_value': None, 'urgency': 'high'
        }

    # group_proposal message ###
    m = re.match(r'group_proposal\s+(.+)', text, re.IGNORECASE)
    if m:
        return {
            'intent': 'group_proposal', 'target': 'Legal',
            'tone': 'formal', 'cc': ['Finance', 'CTO'],
            'include_document': None,
            'concession_term': None, 'concession_value': None, 'urgency': 'normal'
        }

    # walkaway
    if 'walkaway' in text.lower():
        return {
            'intent': 'walkaway', 'target': 'ExecSponsor',
            'tone': 'formal', 'cc': [],
            'include_document': None,
            'concession_term': None, 'concession_value': None, 'urgency': 'normal'
        }

    # address_concern target message ###  (fallback)
    m = re.match(r'address_concern\s+(\w+)\s+(.+)', text, re.IGNORECASE)
    if m:
        return {
            'intent': 'address_concern', 'target': _norm_target(m.group(1)),
            'tone': 'formal', 'cc': [],
            'include_document': None,
            'concession_term': None, 'concession_value': None, 'urgency': 'normal'
        }

    # Default fallback
    return {
        'intent': 'address_concern', 'target': 'Legal',
        'tone': 'formal', 'cc': [],
        'include_document': None,
        'concession_term': None, 'concession_value': None, 'urgency': 'normal'
    }


# Test the parser
test_texts = [
    'offer_document Legal DPA Our DPA covers all compliance requirements. ###',
    'make_concession Finance price=85000 Reducing price to address budget concerns. ###',
    'escalate_to_exec ExecSponsor We would like to schedule a call. ###',
    'address_concern CTO What integration approach are you proposing? ###',
]
for txt in test_texts:
    parsed = parse_email_action(txt)
    print(f'{txt[:50]:<52} -> {parsed["intent"]} -> {parsed["target"]}')



## CELL 6 — GRPO Reward Function

Key design:
- **3-step rollout**: model action + 2 heuristic follow-ups (CVaR veto, belief propagation need multiple steps)
- **Discounted cumulative**: r1 + 0.6*r2 + 0.36*r3
- **No reward overrides**: natural 5D world-state signal from the environment
- **Anti-gaming**: CTA penalty if no response follows

In [ ]:
from typing import List
import threading, random, numpy as np

HEURISTIC_FOLLOWUPS = [
    {'intent': 'offer_document',  'target': 'Legal',    'include_document': 'DPA',              'tone': 'formal'},
    {'intent': 'offer_document',  'target': 'Finance',  'include_document': 'roi_model',        'tone': 'reassuring'},
    {'intent': 'address_concern','target': 'CTO',       'include_document': 'security_cert',     'tone': 'formal'},
    {'intent': 'offer_document',  'target': 'Legal',    'include_document': 'implementation_timeline', 'tone': 'formal'},
]

_components_log = []
_log_lock = threading.Lock()

class EmailNegotiationRewardFunction:
    """
    3-step rollout GRPO reward function.
    
    The environment's EmailNegotiationCore handles:
    - Buyer stakeholder response generation
    - 3-layer reward extraction (keyword -> LLM structured features -> code)
    - Anti-gaming validation
    - progress_score computation (S^0.4 * D^0.2 * G^0.2 * R^0.2)
    
    This class provides:
    - Text -> EmailAction parsing
    - 3-step discounted cumulative reward
    - Variance check before training
    """
    def __init__(self, task_id='hostile_acquisition', base_seed=42,
                 n_rollout_steps=3, discount_factor=0.6):
        self.task_id = task_id
        self.base_seed = base_seed
        self.n_rollout_steps = n_rollout_steps
        self.discount = discount_factor

    def __call__(self, prompts, completions, completion_ids=None, **kwargs) -> List[float]:
        rewards = []
        for i, (prompt, completion) in enumerate(zip(prompts, completions)):
            seed = self.base_seed + (i % 50)

            env = EmailNegotiationCore(
                scenario_type=self.task_id,
                use_buyer_llm=False
            )
            env.reset()

            cumulative = 0.0
            weight = 1.0

            # Parse LLM text -> EmailAction dict
            action_dict = parse_email_action(str(completion))

            # Step 1: model's action
            result = env.step(action_dict)
            cumulative += weight * result['reward']
            weight *= self.discount
            components = result.get('reward_breakdown', {})

            # Steps 2-3: heuristic follow-up (mimics continuation of negotiation)
            if not result['done'] and self.n_rollout_steps > 1:
                rng = random.Random(seed + 100)
                for _ in range(self.n_rollout_steps - 1):
                    followup = rng.choice(HEURISTIC_FOLLOWUPS).copy()
                    result = env.step(followup)
                    cumulative += weight * result['reward']
                    weight *= self.discount
                    if result['done']:
                        break

            env.close()
            rewards.append(float(cumulative))

            if components:
                with _log_lock:
                    _components_log.append(components)

        return rewards

reward_fn = EmailNegotiationRewardFunction(task_id='hostile_acquisition', base_seed=42)
reward_fn.__name__ = 'EmailNegotiationRewardFunction'

# Variance check
test_completions = [
    'offer_document Legal DPA Our DPA covers all compliance requirements. ###',
    'make_concession Finance price=85000 We can reduce the price. ###',
    'escalate_to_exec ExecSponsor We need to finalize terms now. ###',
    'address_concern CTO What integration approach are you proposing? ###',
]
test_rewards = reward_fn(prompts=['']*4, completions=test_completions)
print('Reward variance check:')
for i, (comp, r) in enumerate(zip(test_completions, test_rewards)):
    print(f'  completion {i+1}: {r:+.4f} | {comp[:50]}')
print(f'std={np.std(test_rewards):.4f}')
assert np.std(test_rewards) > 0.005, f'CRITICAL: near-zero variance {np.std(test_rewards):.4f}'
print('PASS: reward function has variance')



## CELL 7 — Load Qwen2.5-3B-Instruct with Unsloth 4-bit QLoRA

In [ ]:
import torch
from unsloth import FastLanguageModel
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

MODEL_NAME     = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

print('Loading Qwen2.5-3B-Instruct...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print(f'Model loaded. Vocab: {tokenizer.vocab_size:,}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB used')



## CELL 8 — Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')



## CELL 9 — Build training dataset

All 3 scenarios (aligned / conflicted / hostile_acquisition) to prevent overfitting
to a single difficulty level. Each prompt = situation text + system prompt.

In [ ]:
from datasets import Dataset

def build_chat_prompt(situation_text, system_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': situation_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

SCENARIOS = [
    ('aligned',             10, 42),
    ('conflicted',          20, 52),
    ('hostile_acquisition', 20, 72),
]

all_prompts = []
for scenario, n, base_seed in SCENARIOS:
    count = 0
    for seed in range(base_seed, base_seed + n + 5):
        if count >= n:
            break
        try:
            e = EmailNegotiationCore(scenario_type=scenario, use_buyer_llm=False)
            situation = e.reset()
            e.close()
            prompt_text = situation.get('inbox_summary', '')
            all_prompts.append({
                'prompt': build_chat_prompt(prompt_text, SYSTEM_PROMPT),
                'scenario': scenario
            })
            count += 1
        except Exception as ex:
            print(f'  seed {seed} failed: {ex}')
    print(f'  {scenario}: {count} prompts')

train_dataset = Dataset.from_list(all_prompts)
print(f'Total dataset: {len(train_dataset)} prompts')

# Token count check
tokens = tokenizer(all_prompts[0]['prompt'], return_tensors='pt')
n_tok = tokens.input_ids.shape[1]
print(f'Sample prompt: {n_tok} tokens (max_seq_length={MAX_SEQ_LENGTH})')
assert n_tok < MAX_SEQ_LENGTH - 128, f'Prompt too long: {n_tok} tokens'



## CELL 10 — GRPOConfig

Key settings:
- `max_completion_length=128` (actions are 20-50 tokens; 256 caused clipped_ratio=1.0)
- `###` as stop sequence (token ID from tokenizer)
- `temperature=0.9` (more diversity = better reward variance)

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# ### token from tokenizer
hash_token_ids = tokenizer.encode('###', add_special_tokens=False)
print(f"'###' token IDs: {hash_token_ids}")
STOP_TOKEN_IDS = hash_token_ids if hash_token_ids else [tokenizer.eos_token_id]

training_args = GRPOConfig(
    output_dir='./email_negotiation_grpo_checkpoints',

    # Training schedule
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=50,
    seed=42,

    # Generation — FIXED: 128 tokens max (was 256 in v3.6)
    max_prompt_length=1400,
    max_completion_length=128,
    num_generations=8,
    temperature=0.9,            # Higher = more diversity = better variance

    # GRPO
    epsilon=0.2,
    beta=0.001,

    # Optimization
    learning_rate=5e-6,
    gradient_checkpointing=True,
    fp16=True,

    # Logging
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    report_to='none',
)

print(f'max_completion_length: {training_args.max_completion_length}')
print(f'temperature:           {training_args.temperature}')
print(f'num_generations:       {training_args.num_generations}')
print(f'Expected time T4:     ~{training_args.max_steps * 1.5:.0f} min')



## CELL 11 — Initialize GRPOTrainer

In [ ]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print('GRPOTrainer initialized.')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable:    {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')



## CELL 12 — GRPO Training

Watch for:
- `reward_std` should be > 0.02 throughout training
- `clipped_ratio` should be near 0 (actions are short)
- `mean_terminated_length` should be 12-25 tokens (model stops at ###)
- `reward` should trend upward over steps

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

print('Starting GRPO training...')
print('=' * 60)
print('Model:        Qwen2.5-3B-Instruct-bnb-4bit (Unsloth)')
print('Environment: EmailNegotiationCore (3 scenarios)')
print('Reward:       3-step rollout, S×D×G×R gating, no overrides')
print('Stop token:   ### (forces clean action termination)')
print('=' * 60)

trainer.train()

print('\nTraining complete!')



## CELL 13 — Training Curves & Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract logged metrics from trainer.state.log_history
history = trainer.state.log_history

steps      = [x['step']   for x in history if 'step'   in x]
rewards   = [x['reward'] for x in history if 'reward' in x]
reward_std= [x['reward_std'] for x in history if 'reward_std' in x]
kl        = [x['kl']     for x in history if 'kl'      in x]
loss      = [x['train_loss'] for x in history if 'train_loss' in x]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Reward
ax = axes[0]
ax.plot(steps, rewards, color='steelblue', lw=2)
if len(steps) > 5:
    window = min(5, len(steps)//2)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(steps[window-1:], smoothed, color='navy', lw=2.5, label=f'{window}-step avg')
ax.set_title('Mean Reward per Step')
ax.set_xlabel('Step'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(True, alpha=0.3)

# Reward std
ax = axes[1]
ax.plot(steps, reward_std, color='orange', lw=2)
ax.axhline(0.02, color='red', lw=1, ls='--', label='min viable (0.02)')
ax.set_title('Reward Std Dev (Variance Check)')
ax.set_xlabel('Step'); ax.set_ylabel('Std Dev')
ax.legend(); ax.grid(True, alpha=0.3)

# KL divergence
ax = axes[2]
ax.plot(steps, kl, color='green', lw=2)
ax.set_title('KL Divergence (Beta=0.001)')
ax.set_xlabel('Step'); ax.set_ylabel('KL')
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[3]
ax.plot(steps, loss, color='red', lw=2)
ax.set_title('Training Loss')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

if rewards:
    print(f'\n=== Training Summary ===')
    print(f'Steps:       {len(steps)}')
    print(f'Reward:      {rewards[0]:.4f} (step 0) -> {rewards[-1]:.4f} (step {steps[-1]})')
    print(f'Improvement: {rewards[-1] - rewards[0]:+.4f}')
    print(f'Reward std:  {min(reward_std):.4f} - {max(reward_std):.4f}')
    print('Saved: training_curves.png')



## CELL 14 — Save model

Push to HuggingFace or save locally.

In [ ]:
trainer.save_model('./email_negotiation_grpo_final')
print('Model saved to ./email_negotiation_grpo_final')

# To push to HF (uncomment and add your token):
# model.push_to_hub('YOUR_USERNAME/email-negotiation-qwen2.5-3b-grpo')
# tokenizer.push_to_hub('YOUR_USERNAME/email-negotiation-qwen2.5-3b-grpo')



## CELL 15 — Inference: test trained model on held-out scenario

Run the trained model on a new scenario and inspect the email thread.

In [ ]:
from unsloth import FastLanguageModel

# Load trained model
trained_model, trained_tokenizer = FastLanguageModel.from_pretrained(
    './email_negotiation_grpo_final',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(trained_model)

# New scenario
env = EmailNegotiationCore(scenario_type='conflicted', use_buyer_llm=False)
obs = env.reset()

print('=== Conflicted Scenario — Trained Model Inference ===\n')
print(f'Initial inbox:\n{obs["inbox_summary"]}\n')

for turn in range(6):
    prompt = tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs['inbox_summary']},
    ], tokenize=False, add_generation_prompt=True)
    
    inputs = trained_tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
        eos_token_id=trained_tokenizer.encode('###', add_special_tokens=False)[0],
    )
    completion = trained_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    action_dict = parse_email_action(completion)
    print(f'Turn {turn+1}: {action_dict["intent"]} -> {action_dict["target"]}')
    print(f'  Raw: {completion[:80].strip()}')
    
    result = env.step(action_dict)
    print(f'  Reward: {result["reward"]:+.3f} | Progress: {result["progress_score"]:.3f} | Stage: {result["deal_stage"]}')
    obs = result
    
    if result['done']:
        print(f'\n  TERMINAL: {env.get_state().get("terminal_outcome")}')
        break
    print()



## CELL 16 — Deploy to HuggingFace Spaces (openenv push)

After training, push to HF Spaces:

```bash
cd /content/deal_room/email_negotiation
openenv push
```

The `openenv.yaml` and `pyproject.toml` are already configured.
The `server/app.py` wraps `EmailNegotiationEnvironment` with `create_fastapi_app`.

**Remote training** — connect the notebook to the running Space:

```python
from email_negotiation.client import EmailNegotiationEnv

SPACE_URL = 'https://YOUR_USERNAME-email-negotiation.hf.space'

def openenv_reward_remote(completions, **kwargs):
    rewards = []
    with EmailNegotiationEnv(base_url=SPACE_URL).sync() as env:
        env.reset()
        for completion in completions:
            action = parse_email_action(str(completion))
            result = env.step(action)
            rewards.append(result.reward)
    return rewards
```